### DrugBank XML에서 약물-타깃 유전자 테이블 만들기

이 노트북은 DrugBank full database XML 파일을 읽어서, 각 약물이 어떤 target gene에 작용하는지 정리한 `targets.txt` 파일을 생성한다.

최종 출력 파일은 다음과 같은 형태를 가진다.

| DrugBank_ID | drug_name | drug_synonyms | target_id | target_name | gene_name | gene_synonyms | gene_identifiers | organism |
|---|---|---|---|---|---|---|---|---|

이 파일은 이후 LINCS metadata의 `cmap_name` 또는 `pert_id`와 연결하여, chemical perturbation의 target gene set `U′`를 만들 때 사용한다.

In [75]:
import os
import re
import time
import numpy as np
import pandas as pd
import sqlite3


In [76]:
LINCS_DIR = "./lincs/"
OUTPUT_DIR = "./chembl/"
DATA = "./data/"

In [77]:
METADATA = os.path.join(LINCS_DIR, "metadata_X.csv")
CHEMBL_TARGETS = os.path.join(OUTPUT_DIR, 'targets')
GENE_INFO = os.path.join(DATA,"geneinfo_beta.txt")
CHEMBL_DB = ".\data\chembl_36\chembl_36_sqlite\chembl_36.db"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [78]:
metadata_X = pd.read_csv(METADATA, low_memory = False)
metadata_X = metadata_X.rename(columns={"Unnamed: 0": "sample_id"})
metadata_X[["sample_id", "pert_id", "cell_iname", "cmap_name"]].head()

,sample_id,pert_id,cell_iname,cmap_name
0,ABY001_A549_XH:BRD-A61304759:0.625:24,BRD-A61304759,A549,tanespimycin
1,ABY001_A549_XH:BRD-A61304759:10:24,BRD-A61304759,A549,tanespimycin
2,ABY001_A549_XH:BRD-A61304759:2.5:24,BRD-A61304759,A549,tanespimycin
3,ABY001_A549_XH:BRD-A90490067:0.625:24,BRD-A90490067,A549,fulvestrant
4,ABY001_A549_XH:BRD-A90490067:10:24,BRD-A90490067,A549,fulvestrant


In [79]:
conn = sqlite3.connect(CHEMBL_DB)

In [80]:
drug_name_sql = """
SELECT
    md.chembl_id AS molecule_chembl_id,
    md.pref_name AS drug_name
FROM molecule_dictionary md
WHERE md.pref_name IS NOT NULL

UNION

SELECT
    md.chembl_id AS molecule_chembl_id,
    ms.synonyms AS drug_name
FROM molecule_dictionary md
JOIN molecule_synonyms ms
    ON md.molregno = ms.molregno
WHERE ms.synonyms IS NOT NULL
"""

chembl_drug_names = pd.read_sql_query(drug_name_sql, conn)
chembl_drug_names.head()

,molecule_chembl_id,drug_name
0,CHEMBL10,SB-203580
1,CHEMBL100,(-)-cromakalim
2,CHEMBL100,BRL-38227
3,CHEMBL100,LEVCROMAKALIM
4,CHEMBL100,Levcromakalim


In [81]:
def normalize_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", "", x)
    return x

In [82]:
chembl_drug_names["drug_name_norm"] = chembl_drug_names["drug_name"].apply(normalize_name)

chembl_drug_names = chembl_drug_names[
    chembl_drug_names["drug_name_norm"] != ""
].drop_duplicates()

chembl_drug_names.shape

(150117, 3)

In [83]:
metadata_X["cmap_name_norm"] = metadata_X["cmap_name"].apply(normalize_name)

metadata_with_chembl = metadata_X.merge(
    chembl_drug_names[["molecule_chembl_id", "drug_name", "drug_name_norm"]],
    left_on="cmap_name_norm",
    right_on="drug_name_norm",
    how="left",
)

In [84]:
metadata_with_chembl = (
    metadata_with_chembl
    .sort_values(["sample_id", "molecule_chembl_id"])
    .drop_duplicates(subset=["sample_id"], keep="first")
    .reset_index(drop=True)
)

In [85]:
print("Total samples:", len(metadata_with_chembl))
print("Mapped samples:", metadata_with_chembl["molecule_chembl_id"].notna().sum())
print("Mapping rate:", metadata_with_chembl["molecule_chembl_id"].notna().mean())

Total samples: 35149
Mapped samples: 31299
Mapping rate: 0.8904663006059916


In [86]:
metadata_with_chembl[
    metadata_with_chembl["molecule_chembl_id"].isna()
][["pert_id", "cmap_name"]].drop_duplicates().head(30)

,pert_id,cmap_name
61,BRD-A19037878,BRD-A19037878
436,BRD-K41895714,AS-605240
478,BRD-K76674262,BRD-K76674262
559,BRD-K06854232,AM-580
574,BRD-K05804044,AZ-628
586,BRD-K63750851,BRD-K63750851
1603,BRD-K83508485,FK-888
1606,BRD-K74305673,BRD-K74305673
1612,BRD-A39415247,BRD-A39415247
1618,BRD-K01567962,pyrazolanthrone


In [87]:
mechanism_sql = """
SELECT DISTINCT
    md.chembl_id AS molecule_chembl_id,
    td.chembl_id AS target_chembl_id,
    cs.component_synonym AS gene_symbol,
    dm.mechanism_of_action,
    dm.action_type
FROM drug_mechanism dm
JOIN molecule_dictionary md
    ON dm.molregno = md.molregno
JOIN target_dictionary td
    ON dm.tid = td.tid
JOIN target_components tc
    ON td.tid = tc.tid
JOIN component_synonyms cs
    ON tc.component_id = cs.component_id
WHERE td.organism = 'Homo sapiens'
  AND td.target_type = 'SINGLE PROTEIN'
  AND cs.syn_type = 'GENE_SYMBOL'
"""

mechanism_targets = pd.read_sql_query(mechanism_sql, conn)
mechanism_targets.head()

,molecule_chembl_id,target_chembl_id,gene_symbol,mechanism_of_action,action_type
0,CHEMBL404271,CHEMBL2074,MGAM,Maltase-glucoamylase inhibitor,INHIBITOR
1,CHEMBL1561,CHEMBL2074,MGAM,Maltase-glucoamylase inhibitor,INHIBITOR
2,CHEMBL19449,CHEMBL1827,PDE5A,Phosphodiesterase 5A inhibitor,INHIBITOR
3,CHEMBL779,CHEMBL1827,PDE5A,Phosphodiesterase 5A inhibitor,INHIBITOR
4,CHEMBL1339,CHEMBL1827,PDE5A,Phosphodiesterase 5A inhibitor,INHIBITOR


In [88]:
'''activity_sql = """
SELECT DISTINCT
    md.chembl_id AS molecule_chembl_id,
    td.chembl_id AS target_chembl_id,
    cs.component_synonym AS gene_symbol
FROM activities act
JOIN assays ass
    ON act.assay_id = ass.assay_id
JOIN molecule_dictionary md
    ON act.molregno = md.molregno
JOIN target_dictionary td
    ON ass.tid = td.tid
JOIN target_components tc
    ON td.tid = tc.tid
JOIN component_synonyms cs
    ON tc.component_id = cs.component_id
WHERE td.organism = 'Homo sapiens'
  AND td.target_type = 'SINGLE PROTEIN'
  AND cs.syn_type = 'GENE_SYMBOL'
  AND act.pchembl_value >= 6
"""

activity_targets = pd.read_sql_query(activity_sql, conn)
activity_targets.head()'''

'activity_sql = """\nSELECT DISTINCT\n    md.chembl_id AS molecule_chembl_id,\n    td.chembl_id AS target_chembl_id,\n    cs.component_synonym AS gene_symbol\nFROM activities act\nJOIN assays ass\n    ON act.assay_id = ass.assay_id\nJOIN molecule_dictionary md\n    ON act.molregno = md.molregno\nJOIN target_dictionary td\n    ON ass.tid = td.tid\nJOIN target_components tc\n    ON td.tid = tc.tid\nJOIN component_synonyms cs\n    ON tc.component_id = cs.component_id\nWHERE td.organism = \'Homo sapiens\'\n  AND td.target_type = \'SINGLE PROTEIN\'\n  AND cs.syn_type = \'GENE_SYMBOL\'\n  AND act.pchembl_value >= 6\n"""\n\nactivity_targets = pd.read_sql_query(activity_sql, conn)\nactivity_targets.head()'

In [89]:
'''mech_mols = set(mechanism_targets["molecule_chembl_id"])

activity_fallback = activity_targets[
    ~activity_targets["molecule_chembl_id"].isin(mech_mols)
].copy()

chembl_targets = pd.concat(
    [
        mechanism_targets[["molecule_chembl_id", "gene_symbol"]],
        activity_fallback[["molecule_chembl_id", "gene_symbol"]],
    ],
    ignore_index=True,
).drop_duplicates()'''

'mech_mols = set(mechanism_targets["molecule_chembl_id"])\n\nactivity_fallback = activity_targets[\n    ~activity_targets["molecule_chembl_id"].isin(mech_mols)\n].copy()\n\nchembl_targets = pd.concat(\n    [\n        mechanism_targets[["molecule_chembl_id", "gene_symbol"]],\n        activity_fallback[["molecule_chembl_id", "gene_symbol"]],\n    ],\n    ignore_index=True,\n).drop_duplicates()'

In [90]:
chembl_targets = mechanism_targets[["molecule_chembl_id", "gene_symbol"]].drop_duplicates()

In [91]:
chembl_target_lists = (
    chembl_targets
    .groupby("molecule_chembl_id")["gene_symbol"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
    .rename(columns={"gene_symbol": "chembl_target_genes"})
)

chembl_target_lists.head()

,molecule_chembl_id,chembl_target_genes
0,CHEMBL1009,[DRD3]
1,CHEMBL101253,"[CSF1R, KIT]"
2,CHEMBL1014,[AGTR1]
3,CHEMBL1016,[AGTR1]
4,CHEMBL1017,[AGTR1]


In [92]:
metadata_targets = metadata_with_chembl.merge(
    chembl_target_lists,
    on="molecule_chembl_id",
    how="left",
)

metadata_targets["chembl_target_genes"] = metadata_targets["chembl_target_genes"].apply(
    lambda x: x if isinstance(x, list) else []
)

metadata_targets["n_chembl_targets"] = metadata_targets["chembl_target_genes"].apply(len)

In [93]:
metadata_targets[
    ["sample_id", "cell_iname", "cmap_name", "molecule_chembl_id", "chembl_target_genes", "n_chembl_targets"]
].head()

,sample_id,cell_iname,cmap_name,molecule_chembl_id,chembl_target_genes,n_chembl_targets
0,ABY001_A549_XH:BRD-A61304759:0.625:24,A549,tanespimycin,CHEMBL109480,[],0
1,ABY001_A549_XH:BRD-A61304759:10:24,A549,tanespimycin,CHEMBL109480,[],0
2,ABY001_A549_XH:BRD-A61304759:2.5:24,A549,tanespimycin,CHEMBL109480,[],0
3,ABY001_A549_XH:BRD-A90490067:0.625:24,A549,fulvestrant,CHEMBL1358,[ESR1],1
4,ABY001_A549_XH:BRD-A90490067:10:24,A549,fulvestrant,CHEMBL1358,[ESR1],1


In [94]:
print("Total samples:", len(metadata_targets))
print("Samples with mechanism targets:", (metadata_targets["n_chembl_targets"] > 0).sum())
print("Mechanism target mapping rate:", (metadata_targets["n_chembl_targets"] > 0).mean())

Total samples: 35149
Samples with mechanism targets: 10718
Mechanism target mapping rate: 0.3049304389883069


In [95]:
gene_info = pd.read_csv(GENE_INFO, sep="\t", low_memory=False)

gene_info_use = gene_info[["gene_id", "gene_symbol"]].dropna().copy()
gene_info_use["gene_id"] = gene_info_use["gene_id"].astype(int)
gene_info_use["gene_symbol"] = gene_info_use["gene_symbol"].astype(str)
gene_info_use = gene_info_use.reset_index(drop=True)

lincs_gene_ids = gene_info_use["gene_id"].to_numpy()
lincs_gene_symbols = gene_info_use["gene_symbol"].to_numpy()

symbol_to_pos = {
    symbol: idx
    for idx, symbol in enumerate(lincs_gene_symbols)
}

n_genes = len(lincs_gene_symbols)

In [96]:
def map_targets_to_lincs_positions(target_genes):
    mapped_genes = []
    positions = []
    
    for gene in target_genes:
        if gene in symbol_to_pos:
            mapped_genes.append(gene)
            positions.append(symbol_to_pos[gene])
    
    return mapped_genes, positions

In [97]:
mapped = metadata_targets["chembl_target_genes"].apply(map_targets_to_lincs_positions)

metadata_targets["target_genes_lincs"] = mapped.apply(lambda x: x[0])
metadata_targets["target_positions_lincs"] = mapped.apply(lambda x: x[1])
metadata_targets["n_lincs_targets"] = metadata_targets["target_positions_lincs"].apply(len)

metadata_mapped = metadata_targets[
    metadata_targets["n_lincs_targets"] > 0
].copy()

metadata_mapped.shape

(10670, 48)

In [98]:
def make_u_prime_mask(target_positions, n_genes):
    mask = np.zeros(n_genes, dtype=np.int8)
    mask[target_positions] = 1
    return mask

In [99]:
metadata_mapped["U_prime"] = metadata_mapped["target_positions_lincs"].apply(
    lambda pos: make_u_prime_mask(pos, n_genes)
)

U_prime_matrix = np.vstack(metadata_mapped["U_prime"].values)

U_prime_matrix.shape

(10670, 12328)

In [100]:
metadata_save = metadata_mapped.drop(columns=["U_prime"]).copy()

for col in ["chembl_target_genes", "target_genes_lincs", "target_positions_lincs"]:
    metadata_save[col] = metadata_save[col].apply(
        lambda x: ",".join(map(str, x)) if isinstance(x, list) else ""
    )

metadata_save.to_csv(
    os.path.join(OUTPUT_DIR, "metadata_X_with_chembl_targets.csv"),
    index=False,
)

In [101]:
np.savez_compressed(
    os.path.join(OUTPUT_DIR, "U_prime_chembl_lincs_positions.npz"),
    U_prime=U_prime_matrix,
    sample_ids=metadata_mapped["sample_id"].astype(str).to_numpy(),
    pert_ids=metadata_mapped["pert_id"].astype(str).to_numpy(),
    cmap_names=metadata_mapped["cmap_name"].astype(str).to_numpy(),
    cell_inames=metadata_mapped["cell_iname"].astype(str).to_numpy(),
    chembl_molecule_ids=metadata_mapped["molecule_chembl_id"].astype(str).to_numpy(),
    gene_ids=lincs_gene_ids,
    gene_symbols=lincs_gene_symbols,
)

In [103]:
metadata_save.head(3)

,sample_id,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_itime,pert_time,pert_time_unit,cell_mfc_name,...,pert_dose_num,cmap_name_norm,molecule_chembl_id,drug_name,drug_name_norm,chembl_target_genes,n_chembl_targets,target_genes_lincs,target_positions_lincs,n_lincs_targets
3,ABY001_A549_XH:BRD-A90490067:0.625:24,b15,0.66,0.625,uM,0.66 uM,24 h,24.0,h,A549,...,0.625,fulvestrant,CHEMBL1358,FULVESTRANT,fulvestrant,ESR1,1,ESR1,4182,1
4,ABY001_A549_XH:BRD-A90490067:10:24,b15,10.00,10.000,uM,10 uM,24 h,24.0,h,A549,...,10.000,fulvestrant,CHEMBL1358,FULVESTRANT,fulvestrant,ESR1,1,ESR1,4182,1
5,ABY001_A549_XH:BRD-A90490067:2.5:24,b15,2.50,2.500,uM,2.5 uM,24 h,24.0,h,A549,...,2.500,fulvestrant,CHEMBL1358,FULVESTRANT,fulvestrant,ESR1,1,ESR1,4182,1
